In [ ]:
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install lightgbm -q

In [ ]:
train_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/train_dataset_v3.csv"

train_df = pd.read_csv(train_path)

print(train_df.shape)

train_df.head()

(242965, 88)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,yaw_rms,yaw_skewness,yaw_kurtosis,yaw_q25,yaw_q75,yaw_iqr,driver,trip,road_type,behavior
0,0.062950,0.037494,0.001406,0.012961,0.178804,0.058821,0.073110,0.994585,0.711390,0.033116,...,0.026448,0.142478,-1.728146,-0.03325,-0.01400,0.01925,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
1,0.060696,0.034344,0.001179,0.012961,0.144686,0.056502,0.069598,0.821860,0.113281,0.033116,...,0.026217,0.081806,-1.758558,-0.03325,-0.01400,0.01925,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
2,0.060563,0.034495,0.001190,0.012961,0.144686,0.056502,0.069556,0.807773,0.093973,0.033116,...,0.025936,0.018867,-1.769954,-0.03325,-0.01375,0.01950,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
3,0.059966,0.034357,0.001180,0.012961,0.144686,0.053694,0.068969,0.863860,0.203253,0.033116,...,0.025623,-0.041783,-1.764851,-0.03325,-0.01300,0.02025,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
4,0.059171,0.033793,0.001142,0.012961,0.144686,0.053694,0.068001,0.933059,0.452223,0.033116,...,0.025261,-0.102163,-1.749063,-0.03225,-0.01300,0.01925,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE


In [ ]:
target_column = "behavior"

feature_columns = [

    col

    for col in train_df.columns

    if col not in [

        "behavior",
        "driver",
        "trip",
        "road_type"

    ]

]

X = train_df[feature_columns]

y = train_df[target_column]

print(X.shape)

(242965, 84)


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(y)

print(label_encoder.classes_)

['AGGRESSIVE' 'DROWSY' 'NORMAL']


In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    objective="multiclass",
    random_state=42
)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
param_dist = {

    "n_estimators":[
        100,
        200,
        300,
        500
    ],

    "learning_rate":[
        0.01,
        0.03,
        0.05,
        0.1
    ],

    "num_leaves":[
        15,
        31,
        63,
        127
    ],

    "max_depth":[
        -1,
        5,
        10,
        20
    ],

    "subsample":[
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "colsample_bytree":[
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "min_child_samples":[
        10,
        20,
        30,
        50
    ]

}

In [ ]:
groups = train_df["trip"]

print(groups.nunique())

32


In [ ]:
from sklearn.model_selection import GroupKFold

group_kfold = GroupKFold(
    n_splits=6
)

In [ ]:
param_dist = {

    "n_estimators": [200,300,500,700,1000],

    "learning_rate": [0.005,0.01,0.03,0.05,0.1],

    "num_leaves": [15,31,63,127,255],

    "max_depth": [-1,5,10,20,30],

    "min_child_samples": [5,10,20,30,50],

    "subsample": [0.7,0.8,0.9,1.0],

    "colsample_bytree": [0.7,0.8,0.9,1.0],

    "reg_alpha": [0,0.1,0.5,1],

    "reg_lambda": [0,0.1,0.5,1]

}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(

    estimator=lgbm,

    param_distributions=param_dist,

    n_iter=50,

    scoring="f1_weighted",

    cv=group_kfold,

    random_state=42,

    verbose=2,

    n_jobs=-1

)

In [1]:
search.fit(
    X,
    y,
    groups=groups
)

NameError: name 'search' is not defined